# 13 Data Modeling — Create Facts and Dimensions from Staging

This notebook is a practical, executable lesson. Each step follows **Context → Source Problem → Detect → Fix → Verify → Business Result → Exercise**.

## 1. The critical boundary

Facts and dimensions are modeled outputs. They are NOT source data. The source had transactions, customer registry, product catalog and stores; modeling now turns them into analytical structures.

## 2. Define grain first

Fact grain: one row per sales transaction line in this teaching model. Every measure must respect that grain.

In [ ]:
import pandas as pd
sales=pd.read_csv('../11_staging_layer/data/stg_sales.csv')
print('candidate grain:',sales['transaction_id'].nunique(),'transactions')

## 3. Create Dim Customer

Create a warehouse dimension from the customer source.

In [ ]:
customers=pd.read_csv('../11_staging_layer/data/stg_customer.csv')
dim_customer=customers.copy()
dim_customer.insert(0,'customer_key',range(1,len(dim_customer)+1))
dim_customer.to_csv('data/dim_customer.csv',index=False)
dim_customer.head()

## 4. Create Dim Product

Create a warehouse product dimension from the product catalog source.

In [ ]:
products=pd.read_csv('../11_data_transformation/data/stg_product.csv') if False else pd.read_csv('../11_staging_layer/data/stg_product.csv')
dim_product=products.copy(); dim_product.insert(0,'product_key',range(1,len(dim_product)+1))
dim_product.to_csv('data/dim_product.csv',index=False)

## 5. Create Dim Store

Create a warehouse store dimension from the store source.

In [ ]:
stores=pd.read_csv('../05_data_sources/source_data/csv/store_locations.csv')
dim_store=stores.copy(); dim_store.insert(0,'store_key',range(1,len(dim_store)+1))
dim_store.to_csv('data/dim_store.csv',index=False)

## 6. Create Dim Date

Date dimension is generated because date analysis requires calendar attributes.

In [ ]:
dates=pd.DataFrame({'date_id':sorted(sales['date_id'].astype(str).unique())})
dates['date']=pd.to_datetime(dates['date_id'],format='%Y%m%d')
dates['year']=dates['date'].dt.year; dates['month']=dates['date'].dt.month; dates['month_name']=dates['date'].dt.month_name(); dates['weekday']=dates['date'].dt.day_name(); dates['quarter']=dates['date'].dt.quarter
dates.insert(0,'date_key',range(1,len(dates)+1)); dates.to_csv('data/dim_date.csv',index=False)

## 7. Create Fact Sales

Replace business keys with surrogate dimension keys and retain measures.

In [ ]:
fact=sales.merge(dim_customer[['customer_id','customer_key']],on='customer_id',validate='many_to_one').merge(dim_product[['product_id','product_key']],on='product_id',validate='many_to_one').merge(dim_store[['store_id','store_key']],on='store_id',validate='many_to_one')
fact=fact[['transaction_id','customer_key','product_key','store_key','date_id','quantity','unit_price','discount_rate','sales_amount','cost_amount','profit']]
fact.to_csv('data/fact_sales.csv',index=False)
print(fact.head())

## 8. Star schema query with DuckDB

Now the star schema exists, so dimensional queries make sense.

In [ ]:
import duckdb
print(duckdb.sql("SELECT p.category, SUM(f.sales_amount) sales FROM read_csv_auto('data/fact_sales.csv') f JOIN read_csv_auto('data/dim_product.csv') p ON f.product_key=p.product_key GROUP BY p.category ORDER BY sales DESC"))

## Exercises

1. Run every code cell and inspect the output.
2. Modify at least one input value to create a data problem.
3. Detect the problem with code.
4. Fix it without hiding the original record.
5. Verify the result and explain the business impact.